In [0]:
reimbursement_avgdailyrevenue_path=dbutils.widgets.get("reimbursement_avgdailyrevenue_path")
aggregatorsdata_path=dbutils.widgets.get("aggregatorsdata_path")
fact_cubeserviceofficetxn_path=dbutils.widgets.get("fact_cubeserviceofficetxn_path")
cubeserviceofficetxnweekendingdate_path=dbutils.widgets.get("cubeserviceofficetxnweekendingdate_path")
office_path=dbutils.widgets.get("office_path")
cubeserviceofficetxnpayor_path=dbutils.widgets.get("cubeserviceofficetxnpayor_path")
cubeserviceofficetxnservicetype_path=dbutils.widgets.get("cubeserviceofficetxnservicetype_path")
cubeserviceofficetxnsourcesystem_path=dbutils.widgets.get("cubeserviceofficetxnsourcesystem_path")

In [0]:

spark.sql(
  f"""
  DROP VIEW IF EXISTS tmp_visits;
  """
)
spark.sql(
  f"""
DROP VIEW IF EXISTS tmp_visits;
  """
)
spark.sql(
  f"""
DROP VIEW IF EXISTS tmp1_revenue;
  """
)
spark.sql(
  f"""
DROP VIEW IF EXISTS tmp2_invoiced;
  """
)
spark.sql(
  f"""
DROP VIEW IF EXISTS tmp_adr_all;
  """
)
spark.sql(
  f"""
DROP VIEW IF EXISTS tally;
  """
)


In [0]:
spark.sql(
  f"""
TRUNCATE TABLE {reimbursement_avgdailyrevenue_path};
 """
)

In [0]:
spark.sql(
  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_visits AS
SELECT SourceVisitId, InvoiceNumber
FROM (
    SELECT 
        sourcevisitid,
        invoicenumber,
        ROW_NUMBER() OVER (
            PARTITION BY sourcevisitid, invoicenumber
            ORDER BY sourcevisitid, invoicenumber
        ) as rnb
    FROM {aggregatorsdata_path}
    WHERE invoicenumber <> ''
)
WHERE rnb = 1;
 """
)

In [0]:
spark.sql(
  f"""

DECLARE OR REPLACE Previous10QuarterStartDate DATE;
 """
)
spark.sql(
  f"""
DECLARE OR REPLACE Previous10QuarterDateKey INT;
 """
)

spark.sql(
  f"""
SET VAR Previous10QuarterStartDate = DATE_TRUNC('QUARTER', ADD_MONTHS(CURRENT_DATE(), -30));
 """
)
spark.sql(
  f"""
SET VAR Previous10QuarterDateKey = CAST(
    DATE_FORMAT(Previous10QuarterStartDate, 'yyyyMMdd') AS INT
);
 """
)

In [0]:
spark.sql(
  f"""

CREATE OR REPLACE TEMPORARY VIEW tmp1_revenue AS
SELECT 
    o.reimbursementofficeabbreviation,
    o.officeabbreviation,
    p.PayorName,
    fin.WeekEndingDate AS Date,
    SUM(f.CurrentPeriodTotalBilled) AS Total_Billed,
    f.VisitID,
    f.OfficeKey,
    f.PayorKey
FROM {fact_cubeserviceofficetxn_path} f
LEFT JOIN {cubeserviceofficetxnweekendingdate_path} fin
    ON f.WeekEndingDateKey = fin.WeekEndingDateKey
LEFT JOIN {office_path} o
    ON f.OfficeKey = o.officekey
LEFT JOIN {cubeserviceofficetxnpayor_path} p
    ON f.PayorKey = p.PayorKey
LEFT JOIN {cubeserviceofficetxnservicetype_path} st
    ON f.servicetypekey = st.ServiceTypeKey
LEFT JOIN {cubeserviceofficetxnsourcesystem_path}  ss
    ON f.sourcesystemkey = ss.sourcesystemkey
WHERE f.ApprovalKey = 10  -- Only approved transactions
  AND fin.WeekEndingDateKey >= Previous10QuarterDateKey
  AND ss.sourcesystemname IN ('BEARS', 'HCHB Staging DB', 'CubHub Staging DB')
GROUP BY 
    o.reimbursementofficeabbreviation,
    o.officeabbreviation,
    p.PayorName,
    fin.WeekEndingDate,
    f.VisitID,
    f.OfficeKey,
    f.PayorKey;
 """
)

spark.sql(
  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp2_invoiced AS
SELECT t1.*
FROM tmp1_revenue t1
WHERE EXISTS (
    SELECT 1 
    FROM tmp_visits tmp
    WHERE tmp.SourceVisitId = t1.VisitID
);
 """
)

spark.sql(
  f"""
CREATE OR REPLACE TEMPORARY VIEW tally AS
SELECT n FROM (
    SELECT 0 AS n UNION ALL SELECT 1 UNION ALL SELECT 2 UNION ALL SELECT 3
    UNION ALL SELECT 4 UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7
    UNION ALL SELECT 8 UNION ALL SELECT 9 UNION ALL SELECT 10 UNION ALL SELECT 11
    UNION ALL SELECT 12
);
 """
)

spark.sql(
  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_adr_all AS
SELECT 
    t.reimbursementofficeabbreviation,
    t.officeabbreviation,
    t.payorname,
    DATE_ADD(t.Date, CAST(b.n * 7 AS INT)) AS GroupingDate,  
    MAX(CASE WHEN b.n = 0 THEN t.Total_Billed END) AS BilledCurrentWeek,
    SUM(t.Total_Billed) AS Rolling13Weeks,
    SUM(t.Total_Billed) / 91 AS AvgDailyRevenue,  
    t.OfficeKey,
    t.PayorKey
FROM (
    SELECT 
        a.reimbursementofficeabbreviation,
        a.officeabbreviation,
        a.PayorName,
        a.Date,
        SUM(a.Total_Billed) AS Total_Billed,
        a.OfficeKey,
        a.PayorKey
    FROM tmp1_revenue a
    GROUP BY 
        a.reimbursementofficeabbreviation,
        a.officeabbreviation,
        a.PayorName,
        a.OfficeKey,
        a.PayorKey,
        a.Date
) t
CROSS JOIN tally b
GROUP BY 
    t.reimbursementofficeabbreviation,
    t.officeabbreviation,
    t.PayorName,
    DATE_ADD(t.Date, CAST(b.n * 7 AS INT)),
    t.OfficeKey,
    t.PayorKey;
 """
)

spark.sql(
    f"""
    INSERT INTO {reimbursement_avgdailyrevenue_path} (
        reimbursement_office_abbreviation,
        office_abbreviation,
        payor_name,
        grouping_date,
        billed_current_week,
        rolling_13_weeks,
        avg_daily_revenue,
        office_key,
        payor_key,
        date_key
    )
    SELECT 
        reimbursementofficeabbreviation AS reimbursement_office_abbreviation,
        officeabbreviation AS office_abbreviation,
        payorname AS payor_name,
        GroupingDate AS grouping_date,
        BilledCurrentWeek AS billed_current_week,
        Rolling13Weeks AS rolling_13_weeks,
        AvgDailyRevenue AS avg_daily_revenue,
        OfficeKey AS office_key,
        PayorKey AS payor_key,
        CAST(REPLACE(CAST(GroupingDate AS STRING), '-', '') AS INT) AS date_key
    FROM tmp_adr_all;
    """
)



spark.sql(
  f"""

CREATE OR REPLACE TEMPORARY VIEW tally1 AS
SELECT n FROM (
    SELECT 0 AS n UNION ALL SELECT 1 UNION ALL SELECT 2 UNION ALL SELECT 3
    UNION ALL SELECT 4 UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7
    UNION ALL SELECT 8 UNION ALL SELECT 9 UNION ALL SELECT 10 UNION ALL SELECT 11
    UNION ALL SELECT 12
);

 """
)

In [0]:

spark.sql(
  f"""
DECLARE OR REPLACE MGD DATE;
"""
)
spark.sql(
  f"""
DECLARE OR REPLACE GD DATE;
"""
)
spark.sql(
  f"""
SET VAR MGD = (SELECT MAX(grouping_date) FROM {reimbursement_avgdailyrevenue_path});
"""
)
spark.sql(
  f"""
SET VAR GD = DATE_SUB(MGD, 77);  
"""
)
spark.sql(
  f"""
DELETE FROM {reimbursement_avgdailyrevenue_path}
WHERE grouping_date BETWEEN GD AND MGD;
"""
)
spark.sql(
  f"""
DELETE FROM {reimbursement_avgdailyrevenue_path}
WHERE date_key > (
    SELECT DISTINCT WeekEndingDateKey
    FROM {cubeserviceofficetxnweekendingdate_path}
    WHERE ActiveWeekEndingInd= TRUE
);

"""
)